# Emergency Call Speech Emotion Recognition - ML Pipeline

## Problem Statement

Emergency dispatch systems receive thousands of calls daily. Prioritizing calls based on the caller's emotional state can save lives — a caller in **pain** or **stress** may need faster response than an **angry** caller.

This notebook demonstrates a complete ML pipeline for classifying emotions in emergency call audio recordings into 4 classes: **angry**, **drunk**, **painful**, and **stressful**.

### Dataset
- [Speech Emotion Recognition for Emergency Calls](https://www.kaggle.com/datasets/anuvagoyal/speech-emotion-recognition-for-emergency-calls)
- ~326 WAV files from 18 speakers
- 4 emergency sentences (~3 seconds each)
- Both natural and synthetic (pitch-shifted) recordings

### Approach
WAV audio → Mel spectrograms (128×128) → CNN classification

## 1. Setup and Imports

In [ ]:
%matplotlib inline

import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
import warnings
warnings.filterwarnings('ignore')

# Resolve backend directory — try multiple known locations
_cwd = os.getcwd()
_search_paths = [
    os.path.join(_cwd, '..'),                                         # if cwd is backend/notebook/
    os.path.join(_cwd, 'backend'),                                    # if cwd is ml-pipeline-summative/
    os.path.join(_cwd, 'ml-pipeline-summative', 'backend'),           # if cwd is ALU/
    '/Users/elvke/Documents/ALU/ml-pipeline-summative/backend',       # absolute fallback
]

BACKEND_DIR = None
for _p in _search_paths:
    if os.path.isfile(os.path.join(_p, 'data', 'metadata.csv')):
        BACKEND_DIR = os.path.abspath(_p)
        break

assert BACKEND_DIR is not None, f"Could not find backend dir. CWD={_cwd}"
print(f'Backend directory: {BACKEND_DIR}')

# Add backend to path for src imports
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

from src.preprocessing import (
    load_and_extract_features, augment_audio, prepare_dataset,
    SAMPLE_RATE, N_MELS, MAX_LEN, DURATION
)
from src.model import build_model, train_model, evaluate_model, retrain_model
from src.visualization import (
    plot_class_distribution, plot_gender_distribution, plot_type_distribution,
    plot_mel_spectrogram, plot_waveform, plot_confusion_matrix,
    plot_training_history, plot_prediction_probabilities
)
from src.experiments import run_all_experiments, EXPERIMENTS

# visualization.py sets matplotlib backend to "Agg" (headless) for API use
# Override it back to inline for notebook rendering
matplotlib.use('module://matplotlib_inline.backend_inline')

print('All imports successful!')

## 2. Data Loading

In [ ]:
# Load metadata
metadata_path = os.path.join(BACKEND_DIR, 'data', 'metadata.csv')
df = pd.read_csv(metadata_path)
print(f'Total samples: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

In [ ]:
# Dataset summary
print(f'Unique emotions: {df["emotion"].unique()}')
print(f'Unique speakers: {df["speaker"].nunique()}')
print(f'Unique sentences: {df["sentence_code"].nunique()}')
print(f'\nEmotion distribution:')
print(df['emotion'].value_counts())
print(f'\nGender distribution:')
print(df['gender'].value_counts())
print(f'\nNatural vs Synthetic:')
print(df['type'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig = plot_class_distribution(df)
plt.show()

In [ ]:
# Gender distribution per emotion
fig = plot_gender_distribution(df)
plt.show()

In [ ]:
# Natural vs Synthetic
fig = plot_type_distribution(df)
plt.show()

In [ ]:
# Play sample audio from each emotion
emotions = ['angry', 'drunk', 'painful', 'stressful']
for emotion in emotions:
    sample = df[df['emotion'] == emotion].iloc[0]
    filepath = os.path.join(BACKEND_DIR, sample['filepath'])
    print(f'\n--- {emotion.upper()} ---')
    print(f'File: {sample["filename"]}')
    print(f'Sentence: {sample["sentence"]}')
    print(f'Speaker: {sample["speaker"]}, Gender: {sample["gender"]}, Type: {sample["type"]}')
    ipd.display(ipd.Audio(filepath, rate=SAMPLE_RATE))

In [ ]:
# Sample waveforms and spectrograms per emotion
fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for i, emotion in enumerate(emotions):
    sample = df[df['emotion'] == emotion].iloc[0]
    filepath = os.path.join(BACKEND_DIR, sample['filepath'])
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, duration=DURATION)
    
    # Waveform
    librosa.display.waveshow(audio, sr=sr, ax=axes[i, 0], color='#3B82F6')
    axes[i, 0].set_title(f'{emotion.capitalize()} - Waveform')
    axes[i, 0].set_xlabel('Time (s)')
    
    # Spectrogram
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    img = librosa.display.specshow(log_mel, sr=sr, hop_length=512,
                                    x_axis='time', y_axis='mel', ax=axes[i, 1])
    axes[i, 1].set_title(f'{emotion.capitalize()} - Mel Spectrogram')

plt.tight_layout()
plt.show()

## 4. Preprocessing

We convert each WAV file to a 128×128 log-mel spectrogram, treating it as a grayscale image for the CNN.

**Steps:**
1. Load audio at 22050 Hz, clip to 3 seconds
2. Compute mel spectrogram (128 mel bands)
3. Convert to log scale (dB)
4. Pad/truncate to 128 time frames
5. Normalize to [0, 1]

**Data augmentation** (critical for ~326 samples):
- Time stretching (0.9x, 1.1x speed)
- Pitch shifting (±2 semitones)
- Gaussian noise injection

In [ ]:
# Demonstrate preprocessing on a single file
sample = df.iloc[0]
filepath = os.path.join(BACKEND_DIR, sample['filepath'])

# Extract features
features = load_and_extract_features(filepath)
print(f'Input: {sample["filename"]} ({sample["emotion"]})')
print(f'Output shape: {features.shape}')
print(f'Value range: [{features.min():.4f}, {features.max():.4f}]')

# Visualize
plt.figure(figsize=(8, 4))
plt.imshow(features[:, :, 0], aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='Normalized amplitude')
plt.xlabel('Time frames')
plt.ylabel('Mel bands')
plt.title(f'Preprocessed Spectrogram - {sample["emotion"]}')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate augmentation
audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, duration=DURATION)
augmented_list = augment_audio(audio, sr)

print(f'Original audio length: {len(audio)} samples')
print(f'Number of augmented variants: {len(augmented_list)}')

# Visualize original vs augmented
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
labels = ['Original', 'Time Stretch 0.9x', 'Time Stretch 1.1x',
          'Pitch -2', 'Pitch +2', 'Noise Added']
all_audio = [audio] + augmented_list

for idx, (ax, aud, label) in enumerate(zip(axes.flat, all_audio, labels)):
    mel = librosa.feature.melspectrogram(y=aud[:int(SAMPLE_RATE*DURATION)], sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(log_mel, sr=sr, hop_length=512, ax=ax, cmap='viridis')
    ax.set_title(label)

plt.suptitle('Original vs Augmented Spectrograms', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare full dataset (this takes a few minutes)
print('Preparing dataset with augmentation...')
shapes = prepare_dataset(
    metadata_csv=os.path.join(BACKEND_DIR, 'data', 'metadata.csv'),
    test_size=0.2,
    augment=True
)
print(f'\nTrain: X={shapes[0]}, y={shapes[2]}')
print(f'Test:  X={shapes[1]}, y={shapes[3]}')

In [ ]:
# Load prepared data
X_train = np.load(os.path.join(BACKEND_DIR, 'data', 'train', 'X_train.npy'))
y_train = np.load(os.path.join(BACKEND_DIR, 'data', 'train', 'y_train.npy'))
X_test = np.load(os.path.join(BACKEND_DIR, 'data', 'test', 'X_test.npy'))
y_test = np.load(os.path.join(BACKEND_DIR, 'data', 'test', 'y_test.npy'))

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

## 5. Model Architecture

We use a CNN with the following optimization techniques:
- **BatchNormalization** after each convolutional layer
- **Dropout** (0.25 in conv blocks, 0.5 in dense layers) for regularization
- **L2 regularization** on kernel weights
- **GlobalAveragePooling** instead of Flatten (fewer parameters)
- **Adam optimizer** with learning rate scheduling
- **EarlyStopping** to prevent overfitting

In [ ]:
# Build and inspect model
model = build_model(input_shape=X_train.shape[1:])
model.summary()

## 6. Training — 5 Experiments

We run 5 different model configurations to find the best architecture:

| Experiment | Description |
|-----------|-------------|
| exp1_baseline_cnn | 3 conv blocks, Adam lr=0.001 |
| exp2_deeper_l2 | 4 conv blocks + L2 reg, Adam lr=0.0005 |
| exp3_small_aggressive_dropout | Smaller CNN + heavy dropout, Adam lr=0.0001 |
| exp4_large_kernel_sgd | 5×5 kernels, SGD with momentum |
| exp5_separable_conv | SeparableConv2D (lightweight), Adam lr=0.001 |

The best model (by F1 score) is automatically saved as `emotion_classifier.h5`.

In [ ]:
# Run all 5 experiments
experiment_results = run_all_experiments(X_train, y_train, X_test, y_test)

In [ ]:
# Visualize experiment comparison
import json

results = experiment_results['experiments']
exp_names = [r['experiment'].replace('_', '\n') for r in results]
metrics_to_plot = ['accuracy', 'f1_score', 'precision', 'recall']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors = ['#3B82F6', '#EF4444', '#10B981', '#F59E0B', '#8B5CF6']

for ax, metric in zip(axes, metrics_to_plot):
    values = [r[metric] for r in results]
    bars = ax.bar(range(len(results)), values, color=colors)
    ax.set_xticks(range(len(results)))
    ax.set_xticklabels(exp_names, fontsize=7)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_ylim(0, 1)
    
    # Highlight best
    best_idx = values.index(max(values))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=8)

plt.suptitle('Experiment Comparison', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nBest experiment: {experiment_results['best_experiment']}")
print(f"Best F1 score: {experiment_results['best_f1_score']:.4f}")

## 7. Evaluation

We evaluate the best model using 5 metrics:
1. **Accuracy** — overall correctness
2. **Loss** — categorical cross-entropy
3. **F1 Score** (macro) — harmonic mean of precision and recall
4. **Precision** (macro) — proportion of correct positive predictions
5. **Recall** (macro) — proportion of actual positives found

In [ ]:
import tensorflow as tf
from sklearn.metrics import classification_report

# Load the best model
best_model = tf.keras.models.load_model(os.path.join(BACKEND_DIR, 'models', 'emotion_classifier.h5'))

# Evaluate
metrics = evaluate_model(best_model, X_test, y_test)

print('\n=== Evaluation Metrics ===')
print(f'Accuracy:  {metrics["accuracy"]:.4f}')
print(f'Loss:      {metrics["loss"]:.4f}')
print(f'F1 Score:  {metrics["f1_score"]:.4f}')
print(f'Precision: {metrics["precision"]:.4f}')
print(f'Recall:    {metrics["recall"]:.4f}')

In [ ]:
# Confusion matrix
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

fig = plot_confusion_matrix(y_true, y_pred)
plt.show()

In [ ]:
# Per-class classification report
from src.model import EMOTION_LABELS
print(classification_report(y_true, y_pred, target_names=EMOTION_LABELS))

In [ ]:
# Training history of the best experiment
best_exp = experiment_results['best_experiment']
best_result = next(r for r in results if r['experiment'] == best_exp)
history = best_result['training_history']

fig = plot_training_history(history)
plt.show()

## 8. Hyperparameter Tuning

Beyond the 5 experiments above, we also ran a grid search over:
- Learning rate: [0.001, 0.0005, 0.0001]
- Batch size: [8, 16, 32]
- Dropout rate: [0.25, 0.5]

In [ ]:
from src.tuning import grid_search

# Run a smaller grid search (subset to save time)
small_grid = {
    'learning_rate': [0.001, 0.0001],
    'batch_size': [8, 16],
    'dropout_rate': [0.25, 0.5],
}

tuning_results = grid_search(X_train, y_train, X_test, y_test, 
                              param_grid=small_grid, epochs=20)

In [ ]:
# Display tuning results as a table
tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df[['learning_rate', 'batch_size', 'dropout_rate', 
                        'val_accuracy', 'val_loss', 'best_epoch']]
tuning_df = tuning_df.sort_values('val_accuracy', ascending=False)
tuning_df

## 9. Prediction Demo

In [ ]:
from src.prediction import predict, CLASS_LABELS

# Predict on random test samples
np.random.seed(42)
sample_indices = np.random.choice(len(df), 5, replace=False)

for idx in sample_indices:
    row = df.iloc[idx]
    filepath = os.path.join(BACKEND_DIR, row['filepath'])
    
    result = predict(filepath)
    actual = row['emotion']
    predicted = result['class_label']
    correct = '✓' if actual == predicted else '✗'
    
    print(f'{correct} Actual: {actual:<12} Predicted: {predicted:<12} '
          f'Confidence: {result["confidence"]:.1%}')
    
    # Show confidence chart
    fig = plot_prediction_probabilities(
        list(result['probabilities'].values()),
        list(result['probabilities'].keys())
    )
    plt.show()

## 10. Retraining Demo

The retraining pipeline:
1. Load existing model as pretrained base
2. Freeze early convolutional layers
3. Combine new data with 30% of original training data
4. Fine-tune with lower learning rate (0.0001)

In [ ]:
# Simulate retraining with a small subset of test data
# (In production, this would be new uploaded data)
retrain_X = X_test[:10]
retrain_y = np.argmax(y_test[:10], axis=1)  # Integer labels

print(f'Retraining with {len(retrain_X)} samples...')
print(f'Labels: {retrain_y}')

# Get metrics before retraining
print('\n--- Before Retraining ---')
before_metrics = evaluate_model(best_model, X_test, y_test)

# Retrain
retrained_model, after_metrics = retrain_model(
    retrain_X, retrain_y, epochs=10
)

print('\n--- After Retraining ---')
print(f'Accuracy: {before_metrics["accuracy"]:.4f} -> {after_metrics.get("accuracy", "N/A")}')
print(f'F1 Score: {before_metrics["f1_score"]:.4f} -> {after_metrics.get("f1_score", "N/A")}')

## 11. Conclusions

### Summary
- Built an end-to-end ML pipeline for emergency call speech emotion recognition across 4 classes (angry, drunk, painful, stressful)
- Dataset: ~326 WAV files from 18 speakers, expanded to ~1,620 training samples via 5x audio augmentation (time stretch, pitch shift, noise injection)
- Ran **5 experiments** with different CNN architectures and optimizers, selecting the best by macro F1 score
- Best model: **exp4_large_kernel_sgd** (5×5 kernels + SGD with momentum) — **36.8% accuracy, 0.315 F1 score**
- Hyperparameter tuning via grid search (8 combinations) confirmed that higher learning rate (0.001), small batch size (8), and heavy dropout (0.5) work best for this small dataset
- Demonstrated full prediction and retraining pipelines with before/after metric comparison

### Key Findings
- **Angry is the most recognizable emotion** — the model achieves 83% recall on angry samples (15/18 correct), likely because angry speech has distinct high-energy spectral patterns
- **Drunk is the hardest to classify** — only 6% recall (1/16 correct), with most drunk samples misclassified as angry. Drunk speech patterns may overlap heavily with other emotional states
- **Larger kernels (5×5) outperformed standard 3×3** — capturing wider temporal-frequency context in spectrograms gave exp4 the edge over other architectures
- **SGD with momentum generalized better than Adam** on this small dataset — Adam's adaptive learning rates can overfit quickly with limited data
- **Data augmentation was critical** — expanding 270 training samples to ~1,620 via audio-level augmentation prevented severe overfitting, though a significant train-val gap persists across all experiments
- **All models show overfitting** — training accuracy reaches 50-60% while validation stays at 30-37%, indicating the dataset's limited speaker diversity (18 speakers, 4 sentences) is the main bottleneck
- **Hyperparameter tuning confirmed** that smaller batch sizes (8) produce noisier gradients that help escape poor local minima, and higher dropout (0.5) is needed to regularize the small dataset

### Potential Improvements
- **More data**: The single biggest improvement would be collecting recordings from more speakers with more varied sentences — 326 samples across 18 speakers severely limits generalization
- **Transfer learning**: Use pre-trained audio models (e.g., YAMNet, OpenL3, wav2vec 2.0) as feature extractors instead of training a CNN from scratch — these models have learned rich audio representations from millions of samples
- **Multi-feature approach**: Combine mel spectrograms with handcrafted features (MFCCs, chroma, spectral contrast, zero-crossing rate) to give the model complementary views of the audio signal
- **Speaker-independent splits**: Split train/test by speaker ID rather than randomly to better measure real-world generalization — the current random split likely has the same speakers in both sets
- **Class balancing**: Apply class weights or SMOTE-style oversampling to address the slight class imbalance and improve performance on underrepresented emotions
- **Longer training with larger grid**: The best hyperparameter config (LR=0.001, batch=8, dropout=0.5) was still improving at epoch 20, suggesting more epochs and a wider search space could yield better results